In [1]:
# Source: https://www.kaggle.com/datasets/mattop/nhl-draft-hockey-player-data-1963-2022/data

import pandas as pd
df = pd.read_csv('nhldraft.csv')
#get all number one draft picks
elitePicks = df[df['overall_pick'] == 1]
# drop overall pick because we are only calculating number 1
elitePicks = elitePicks.drop(columns='overall_pick')
#dont include picks who never played a game
elitePicks = elitePicks[elitePicks['id'] != 12206]
elitePicks = elitePicks[elitePicks['id'] != 12195]
elitePicks = elitePicks[elitePicks['id'] != 12153]

# entry Juraj Slafkovsky as of 10-15-2026
elitePicks.loc[elitePicks['id'] == 1, 'games_played'] = 289
elitePicks.loc[elitePicks['id'] == 1, 'goals'] = 73
elitePicks.loc[elitePicks['id'] == 1, 'assists'] = 114
elitePicks.loc[elitePicks['id'] == 1, 'points'] = 187
elitePicks.loc[elitePicks['id'] == 1, 'penalties_minutes'] = 189
elitePicks.loc[elitePicks['id'] == 1, 'plus_minus'] = -21

In [2]:
# create retired dataframe
retired=elitePicks.loc[elitePicks['to_year'] != 2022].copy()
retired=retired.loc[retired['to_year'].notnull(), :]
retired['years_active'] = retired['to_year']-retired['year']
# create averages
avg_years_active=retired['years_active'].mean().round(2)
avg_games_played_retired=retired['games_played'].mean().round(2)
avg_goals_retired=retired['goals'].mean().round(2)
avg_assists_retired=retired['assists'].mean().round(2)
avg_points_retired=retired['points'].mean().round(2)
avg_penalty_retired=retired['penalties_minutes'].mean().round(2)
avg_plus_retired=retired['plus_minus'].mean().round(2)
avg_point_shares_retired=retired['point_shares'].mean().round(2)

print('Average Career Stats of Retired #1s')
print('Average Games Played: ',avg_games_played_retired)
print('Average Goals Scored: ',avg_goals_retired)
print('Average Assists: ',avg_assists_retired)
print('Average Points: ',avg_points_retired)
print('Average Penalty Minutes',avg_penalty_retired)
print('Average Plus Minus',avg_plus_retired)
print('Average Point Shares',avg_point_shares_retired)

Average Career Stats of Retired #1s
Average Games Played:  863.81
Average Goals Scored:  254.11
Average Assists:  398.59
Average Points:  652.7
Average Penalty Minutes 760.41
Average Plus Minus 7.08
Average Point Shares 70.94


In [3]:
# create active dataframe
active=elitePicks.loc[elitePicks['to_year'] == 2022].copy()
active['years_active']=active['to_year']-active['year']

# create averages
avg_years_active_a=active['years_active'].mean().round(2)
avg_games_played_a=active['games_played'].mean().round(2)
avg_goals_a=active['goals'].mean().round(2)
avg_assists_a=active['assists'].mean().round(2)
avg_points_a=active['points'].mean().round(2)
avg_penalty_a=active['penalties_minutes'].mean().round(2)
avg_plus_a=active['plus_minus'].mean().round(2)
avg_point_shares_a=active['point_shares'].mean().round(2)

print('Average Career Stats of Active #1s')
print('Average Games Played: ',avg_games_played_a)
print('Average Goals Scored: ',avg_goals_a)
print('Average Assists: ',avg_assists_a)
print('Average Points: ',avg_points_a)
print('Average Penalty Minutes',avg_penalty_a)
print('Average Plus Minus',avg_plus_a)
print('Average Point Shares',avg_point_shares_a)

Average Career Stats of Active #1s
Average Games Played:  701.16
Average Goals Scored:  241.26
Average Assists:  367.21
Average Points:  608.47
Average Penalty Minutes 344.74
Average Plus Minus 34.37
Average Point Shares 82.3


In [17]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

#  Calculate rate metrics
retired['points_per_game'] = retired['points'] / retired['games_played']
retired['goals_per_game'] = retired['goals'] / retired['games_played']
retired['assists_per_game'] = retired['assists'] / retired['games_played']
retired['pim_per_game'] = retired['penalties_minutes'] / retired['games_played']
active['points_per_game'] = active['points'] / active['games_played']
active['goals_per_game'] = active['goals'] / active['games_played']
active['assists_per_game'] = active['assists'] / active['games_played']
active['pim_per_game'] = active['penalties_minutes'] / active['games_played']

# Select pure rate features
rate_features = ['points_per_game', 'goals_per_game', 'assists_per_game', 'pim_per_game']

# Scale the rate features
scaler = StandardScaler()
retired_scaled = scaler.fit_transform(retired[rate_features].values)
active_scaled = scaler.transform(active[rate_features].values)

# Fit the Nearest Neighbors model
nn = NearestNeighbors(n_neighbors=5, metric='euclidean')
nn.fit(retired_scaled)

# Extract Jack Hughes's row position and look up neighbors
hughes_row_position = active.index.get_loc(664)

# Flatten the input array using [0] to ensure the shape is a clean 2D matrix
distances, indices = nn.kneighbors(active_scaled[[hughes_row_position]])

# Extract the 5 matching peers by flattening the multi-dimensional indices list
matched_indices = indices.flatten()
matched_peers = retired.iloc[matched_indices].copy()

# Project his career totals
avg_retired_games = 863.81  # From your previous summary metrics
pred_points_per_game = matched_peers['points_per_game'].mean()
projected_lifetime_points = round(pred_points_per_game * avg_retired_games)

target_player = active[active['player'].str.contains('Jack Hughes', na=False)].iloc[[0]].copy()

print("--- Target Player ---")
display(target_player[['player', 'year', 'points_per_game', 'games_played', 'points', 'position']])
print(f"\nPredicted Lifetime Points (Based on Comps Efficiency): {projected_lifetime_points} points")
print()
print("--- Similar Players ---")
display(matched_peers[['player', 'year', 'points_per_game', 'games_played', 'points', 'position']])



--- Target Player ---


,player,year,points_per_game,games_played,points,position
664,Jack Hughes,2019,0.650602,166.0,108.0,C



Predicted Lifetime Points (Based on Comps Efficiency): 551 points

--- Similar Players ---


,player,year,points_per_game,games_played,points,position
11660,Billy Harris,1972,0.622074,897.0,558.0,RW
6858,Alexandre Daigle,1993,0.530844,616.0,327.0,C
12044,Rejean Houle,1969,0.643533,634.0,408.0,W
10139,Doug Wickenheiser,1980,0.496403,556.0,276.0,C
10709,Dale McCourt,1977,0.898496,532.0,478.0,C


In [19]:
# create season averages with divided by years_active

In [6]:
#as of 10-15-2026
#bedard = {'id':12231, 'player':'Connor Bedard','year':2023, 'team':'Chicago Blackhawks', 'position':'C/LW', 'age':17, 'nationality':'CA', 'games_played':220, 'goals':75,
          #'assists':129,'points':204,'penalties_minutes':148, 'plus_minus':-97}
#elitePicks.loc[len(elitePicks)] = bedard

#celebrini = {'id':12232, 'player':'Macklin Celebrini','year':2024, 'team':'San Jose Sharks', 'position':'C/LW', 'age':18, 'nationality':'CA', 'games_played':156, 'goals':72,
          #'assists':114,'points':186,'penalties_minutes':176, 'plus_minus':-22}
#elitePicks.loc[len(elitePicks)] = celebrini

#schaefer = {'id':12233, 'player':'Matthew Schaefer','year':2025, 'team':'New York Islanders', 'position':'D', 'age':18, 'nationality':'CA', 'games_played':88, 'goals':23,
          #'assists':42,'points':65,'penalties_minutes':40, 'plus_minus':15}
#elitePicks.loc[len(elitePicks)] = schaefer

#mckenna = {'id':12234, 'player':'Gavin McKenna','year':2026, 'team':'Toronto Maple Leafs', 'position':'LW', 'age':18, 'nationality':'CA', 'games_played':7, 'goals':2,
          #'assists':3,'points':5,'penalties_minutes':4, 'plus_minus':-2}
#elitePicks.loc[len(elitePicks)] = mckenna